# Imports

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms, models
from torchvision.datasets import Caltech101

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Load the data

In [10]:
def convert_to_rgb(img):
    return img.convert("RGB")

# Transform pipeline
transform = transforms.Compose([
    # We need to convert grayscale images to RGB so it uses 3 channels
    transforms.Lambda(convert_to_rgb),
    # We need to give them the same size
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


dataset = Caltech101(root="./data", download=False, transform=transform)


num_classes = len(dataset.categories)
print(f"Total images: {len(dataset)}, Total classes: {num_classes}")

# 80/20 Train-Validation Split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

Total images: 8677, Total classes: 101


# Models

## ResNet34

We will use the ResNet model and freeze all layers except the final one, we'll retrain it on our dataset.

In [ ]:
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False # which means that they don't need to be trained

# Unfreeze last layer
for param in model.layer4.parameters():
    param.requires_grad = True 

# Number of features each neuron in the last layer takes
num_ftrs = model.fc.in_features

# We create a new layer to replace the last layer in the original network
# That new layer has each neuron take the same number of features like the old layer (we can't change that)
# Num of neurons = num classes
# Newly created layers have requires_grad=True by default
model.fc = nn.Linear(num_ftrs, num_classes)


model = model.to(device)

In [7]:
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
            
        train_loss = running_loss / total_train
        train_acc = correct_train / total_train * 100
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)
                
        val_loss = val_loss / total_val
        val_acc = correct_val / total_val * 100
        
        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

Now let's train our new last layer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=1e-4
)

# Run training
train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, epochs=10)

Epoch [1/10] | Train Loss: 0.0480, Train Acc: 99.14% | Val Loss: 0.1822, Val Acc: 95.22%
Epoch [2/10] | Train Loss: 0.0155, Train Acc: 99.74% | Val Loss: 0.1969, Val Acc: 94.99%
Epoch [3/10] | Train Loss: 0.0057, Train Acc: 99.96% | Val Loss: 0.1779, Val Acc: 96.26%
Epoch [4/10] | Train Loss: 0.0073, Train Acc: 99.83% | Val Loss: 0.1690, Val Acc: 96.31%
Epoch [5/10] | Train Loss: 0.0075, Train Acc: 99.86% | Val Loss: 0.1792, Val Acc: 95.51%
Epoch [6/10] | Train Loss: 0.0274, Train Acc: 99.51% | Val Loss: 0.2620, Val Acc: 93.72%
Epoch [7/10] | Train Loss: 0.0257, Train Acc: 99.52% | Val Loss: 0.2103, Val Acc: 94.59%
Epoch [8/10] | Train Loss: 0.0240, Train Acc: 99.52% | Val Loss: 0.2210, Val Acc: 94.87%
Epoch [9/10] | Train Loss: 0.0058, Train Acc: 99.93% | Val Loss: 0.2032, Val Acc: 95.28%
Epoch [10/10] | Train Loss: 0.0133, Train Acc: 99.64% | Val Loss: 0.2204, Val Acc: 94.35%


Now let's save the model

In [6]:
torch.save(model.state_dict(), "resnet34_caltech101.pth")
print("Model saved successfully as resnet34_caltech101.pth")

Model saved successfully as resnet34_caltech101.pth


## MobileNetV2

In [4]:
# Load pre-trained MobileNetV2
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# 1. Freeze ALL layers initially
for param in mobilenet.parameters():
    param.requires_grad = False

# 2. Unfreeze the LAST convolutional block
# In MobileNetV2, all conv layers are inside '.features'. The last block is at index [-1].
for param in mobilenet.features[-1].parameters():
    param.requires_grad = True

# 3. Replace the final fully connected layer to match Caltech 101 classes
# MobileNetV2 stores its final linear layer inside a '.classifier' Sequential block at index 1
num_ftrs_mb = mobilenet.classifier[1].in_features
mobilenet.classifier[1] = nn.Linear(num_ftrs_mb, num_classes)

# Move model to the GPU
mobilenet = mobilenet.to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\Cyber/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100.0%


Now let's train the last layer

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer_mb = optim.Adam(
    filter(lambda p: p.requires_grad, mobilenet.parameters()), 
    lr=1e-4
)

# 5. Run the training loop using the function we defined earlier
print("Starting MobileNetV2 Training...")
train_and_evaluate(mobilenet, train_loader, val_loader, criterion, optimizer_mb, epochs=10)

# 6. Save the trained model
torch.save(mobilenet.state_dict(), "mobilenetv2_caltech101.pth")
print("Model saved successfully as mobilenetv2_caltech101.pth")

Starting MobileNetV2 Training...
Epoch [1/10] | Train Loss: 3.5552, Train Acc: 28.30% | Val Loss: 2.6812, Val Acc: 45.51%
Epoch [2/10] | Train Loss: 2.2610, Train Acc: 59.69% | Val Loss: 1.7956, Val Acc: 74.94%
Epoch [3/10] | Train Loss: 1.4950, Train Acc: 79.59% | Val Loss: 1.1691, Val Acc: 84.74%
Epoch [4/10] | Train Loss: 0.9999, Train Acc: 86.69% | Val Loss: 0.8108, Val Acc: 87.73%
Epoch [5/10] | Train Loss: 0.7156, Train Acc: 91.21% | Val Loss: 0.6227, Val Acc: 89.40%
Epoch [6/10] | Train Loss: 0.5464, Train Acc: 93.10% | Val Loss: 0.5064, Val Acc: 90.55%
Epoch [7/10] | Train Loss: 0.4356, Train Acc: 94.32% | Val Loss: 0.4403, Val Acc: 91.30%
Epoch [8/10] | Train Loss: 0.3614, Train Acc: 95.12% | Val Loss: 0.3951, Val Acc: 92.11%
Epoch [9/10] | Train Loss: 0.3082, Train Acc: 95.59% | Val Loss: 0.3579, Val Acc: 92.74%
Epoch [10/10] | Train Loss: 0.2644, Train Acc: 96.41% | Val Loss: 0.3273, Val Acc: 92.63%
Model saved successfully as mobilenetv2_caltech101.pth
